# Submissions V06 and V07 (Nico)

*(V05 is Ludovico's blend of logistic models, Kaggle 0.80193.)*

**Goal:** V03 plus the customer's own seasonality, the only new feature group that helped in `features_nico` (V06), and the same model with V03's threshold (V07), to separate the effect of the features from that of the threshold.

| | V03 (Kaggle 0.804) | V06 |
|---|---|---|
| Model | logistic regression | same |
| Preprocessing | median/mode, signed log, robust scaler, one-hot (min 50) | same |
| Features | every column + month of purchase | same **+ `seasonal`** (3 columns) |
| Feature selection | L1 (`SelectFromModel`, C = 0.05) | same |
| Threshold | best F1 on the out-of-fold predictions of the official folds | same rule |

**`seasonal`:** for a purchase on date *d*, the customer's purchases in the 90 days after the same date 1, 2 and 3 years earlier (`features_nico` §2). Evidence (`features_nico` §3–5): ROC AUC +0.0037 on 8 folds (7/8 up), 0.796 → 0.801 on the official folds (3/3 up), rolling honest F1 +0.0075 (4/6 folds).

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>Setup</font>](#setup)
* [<font color='#E8800A'>1. The pipeline</font>](#pipeline)
* [<font color='#E8800A'>2. Expected scores from the validation</font>](#expected)
* [<font color='#E8800A'>3. Fit on all train, predict test, submit</font>](#submit)
* [<font color='#E8800A'>4. V07: the same model with V03's threshold</font>](#v07)
* [<font color='#E8800A'>5. Publish and upload</font>](#publish)

</div>

# <font color='#E8800A'>Setup</font> <a class="anchor" id="setup"></a>
[Back to TOC](#toc)

`%run` executes `preprocessing_ludovico.ipynb` (about 2 minutes). It does **not** run `features_nico.ipynb` (about 5 minutes of experiments): the pieces needed from it are restated below exactly as they are defined there.

In [1]:
%%capture
%run ./preprocessing_ludovico.ipynb

In [2]:
from sklearn.feature_selection import SelectFromModel

from src.submissions import check_submission, publish, record_kaggle_score, submission_path, submit

ALWAYS_DROP = ["customer_id", "purchase_date", "random_noise"]
SHORT = 90  # the target's horizon in days
X_full = train.drop(columns=[TARGET])  # identifiers and date are needed to build the features, then dropped
X_test_full = test.copy()

# <font color='#E8800A'>1. The pipeline</font> <a class="anchor" id="pipeline"></a>
[Back to TOC](#toc)

**The `seasonal` columns** (as in `features_nico` §2), computed once on train and test together, from the dates only. A window is NaN when it starts before the customer's first purchase in the data, or when it overlaps the period with no rows between train and test.

In [3]:
rows = pd.concat([X_full, X_test_full])[["customer_id", "purchase_date"]]
MISSING = (X_full["purchase_date"].max(), X_test_full["purchase_date"].min())  # no row in between
dates_of = rows.groupby("customer_id")["purchase_date"].apply(lambda s: np.sort(s.to_numpy()))
first_of = rows.groupby("customer_id")["purchase_date"].min()


def same_window(years):
    """Purchases of the customer in the 90 days after the same date `years` years earlier (NaN if unknown)."""
    starts = rows["purchase_date"] - pd.Timedelta(days=365 * years)
    counts = []
    for customer, start in zip(rows["customer_id"], starts):
        end = start + pd.Timedelta(days=SHORT)
        if start < first_of[customer] or (start < MISSING[1] and end > MISSING[0]):
            counts.append(np.nan)
            continue
        dates = dates_of[customer]
        counts.append(np.searchsorted(dates, np.datetime64(end), "right")
                      - np.searchsorted(dates, np.datetime64(start), "right"))
    return pd.Series(counts, index=rows.index, dtype=float)


windows = pd.concat({k: same_window(k) for k in [1, 2, 3]}, axis=1)
SEASONAL = pd.DataFrame({
    "season_last_year": windows[1],
    "season_share_years": (windows > 0).astype(float).where(windows.notna()).mean(axis=1),
    "season_mean_count": windows.mean(axis=1),
})
SEASONAL.groupby(rows.index.isin(X_test_full.index)).describe().T.round(2).rename(columns={False: "train", True: "test"})

train    test
season_last_year   count  4684.00  643.00
                   mean      1.94    1.79
                   std       3.13    1.95
                   min       0.00    0.00
                   25%       0.00    1.00
                   50%       1.00    1.00
                   75%       2.00    2.00
                   max      22.00   11.00
season_share_years count  4684.00  910.00
                   mean      0.66    0.74
                   std       0.39    0.33
                   min       0.00    0.00
                   25%       0.33    0.50
                   50%       0.67    1.00
                   75%       1.00    1.00
                   max       1.00    1.00
season_mean_count  count  4684.00  910.00
                   mean      1.92    1.72
                   std       3.00    1.82
                   min       0.00    0.00
                   25%       0.33    0.67
                   50%       1.00    1.33
                   75%       2.00    2.00
                   max      19.00    9.50

**The pipeline:** month and `seasonal` → V03's linear recipe → L1 selection → logistic regression. `add_features` reads no target, so it can run before the split.

In [4]:
def add_features(df):
    """Month of purchase (features_ludovico) and the seasonal columns (features_nico), then drop the identifiers."""
    df = df.assign(purchase_month=df["purchase_date"].dt.month.astype(str)).join(SEASONAL)
    return df.drop(columns=[c for c in ALWAYS_DROP if c in df.columns])


def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))


l1_selector = SelectFromModel(
    make_pipeline(StandardScaler(), LogisticRegression(l1_ratio=1, solver="saga", C=0.05, max_iter=5000)),
    importance_getter="named_steps.logisticregression.coef_",
)


def v06_pipeline():
    """Logistic regression: month + seasonal -> linear recipe -> L1 selection -> model."""
    steps = linear_recipe(LogisticRegression(max_iter=2000)).set_params(
        prep__num__transform=FunctionTransformer(signed_log1p, feature_names_out="one-to-one")).steps
    return Pipeline([("features", FunctionTransformer(add_features)), *steps[:-1],
                     ("select", clone(l1_selector)), steps[-1]])

# <font color='#E8800A'>2. Expected scores from the validation</font> <a class="anchor" id="expected"></a>
[Back to TOC](#toc)

Same rules as V01–V04, on the **official folds**:
- **final threshold:** best F1 on the out-of-fold predictions of all folds;
- **honest F1:** threshold chosen on folds 1–2, scored on fold 3 (the number logged as `CV F1`).

The ROC AUC must match `features_nico` §4 (0.8014): the assertion checks that the restated pipeline is the same one.

In [5]:
early = np.concatenate([va for _, va in official_folds[:-1]])
last = official_folds[-1][1]

scores, oof = evaluate(v06_pipeline(), official_folds, X_full, y)
covered = oof.notna()
final_threshold, _ = best_threshold(y[covered], oof[covered])
t_early, _ = best_threshold(y.iloc[early], oof.iloc[early])
expected = pd.Series({"ROC AUC": scores["ROC AUC"].mean(),
                      "final threshold": final_threshold,
                      "threshold folds 1-2": t_early,
                      "honest F1 (fold 3)": f1_score(y.iloc[last], oof.iloc[last] >= t_early)})

assert abs(expected["ROC AUC"] - 0.8014) < 5e-4, expected["ROC AUC"]
expected.round(4)

ROC AUC                0.8014
final threshold        0.3500
threshold folds 1-2    0.4600
honest F1 (fold 3)     0.7894
dtype: float64

**About the honest F1:** on fold 3 alone it is lower than V03's (0.789 vs 0.802), because the threshold chosen on folds 1–2 moves from 0.43 to 0.46. On 6 rolling folds, each with its threshold chosen on the folds before it, V06's features give **+0.0075 F1** over V03's (`features_nico` §5). The single-fold number is the noisier of the two.

# <font color='#E8800A'>3. Fit on all train, predict test, submit</font> <a class="anchor" id="submit"></a>
[Back to TOC](#toc)

The pipeline is refitted on **all** clean train rows (the L1 selector picks its columns again), predicts test, and the final threshold turns probabilities into 0/1. `submit` assigns the next free version number and logs it.

In [6]:
final_model = v06_pipeline().fit(X_full, y)
proba = final_model.predict_proba(X_test_full)[:, 1]
prediction = pd.DataFrame({"ID": test.index, TARGET: (proba >= expected["final threshold"]).astype(int)})
check_submission(prediction)

kept = final_model.named_steps["select"].get_support()
names = np.array([n.split("__")[-1] for n in final_model.named_steps["prep"].get_feature_names_out()])
print(f"L1 kept {kept.sum()} of {len(kept)} columns on all train")
print("seasonal columns kept:", [n for n in names[kept] if n.startswith("season_")])

v03 = pd.read_csv(submission_path(3))
pd.Series({"share of 1s (V06)": prediction[TARGET].mean(),
           "share of 1s (V03)": v03[TARGET].mean(),
           "same label as V03": (prediction[TARGET].to_numpy() == v03[TARGET].to_numpy()).mean()}).round(3)

L1 kept 26 of 87 columns on all train
seasonal columns kept: [np.str_('season_last_year'), np.str_('season_mean_count')]


share of 1s (V06)    0.839
share of 1s (V03)    0.736
same label as V03    0.894
dtype: float64

In [7]:
version_v06 = submit(
    prediction,
    notebook="submission_v06_v07_nico",
    model="logistic regression + linear recipe + month + seasonal + L1 selection (C=0.05)",
    cv_f1=expected["honest F1 (fold 3)"],
    notes=(f"V03 + seasonal (purchases in the 90 days after the same date 1-3 years earlier); "
           f"threshold {expected['final threshold']:.2f} from time CV + gap; CV ROC AUC {expected['ROC AUC']:.3f}; "
           f"rolling honest F1 +0.0075 vs V03 features (features_nico §5); {prediction[TARGET].mean():.0%} predicted 1"),
)

V06 saved as submissions\ML_Group_41_Version06.csv and logged:
  Version: 06 | Date: 2026-10-22 | Author: Bosch | Notebook / model: submission_v06_v07_nico · logistic regression + linear recipe + month + seasonal + L1 selection (C=0.05) | CV F1: 0.789 | Notes: V03 + seasonal (purchases in the 90 days after the same date 1-3 years earlier); threshold 0.35 from time CV + gap; CV ROC AUC 0.801; rolling honest F1 +0.0075 vs V03 features (features_nico §5); 84% predicted 1
Next: publish(6) to reserve the version, then upload the file on Kaggle.


# <font color='#E8800A'>4. V07: the same model with V03's threshold</font> <a class="anchor" id="v07"></a>
[Back to TOC](#toc)

**Why:** V06 predicts 84% of 1s, V03 74%. The two models give almost the same probabilities on test (mean 0.625 vs 0.619); the difference comes from the threshold. The out-of-fold F1 curve is **flat between 0.34 and 0.46** (every threshold within ±0.005 of the best), so the rule picked 0.43 for V03 and 0.35 for V06 almost by chance.

A Kaggle score of V06 alone would mix two changes, the features and the threshold. **V07 keeps V06's model and uses V03's threshold (0.43)**, so that:
- V07 vs V03 = effect of the `seasonal` features only (same threshold);
- V06 vs V07 = effect of the threshold only (same model).

In [8]:
V03_THRESHOLD = 0.43  # docs/submissions.md, row 03

curve = pd.Series({t: f1_score(y[covered], oof[covered] >= t) for t in np.round(np.arange(0.30, 0.52, 0.02), 2)})
pd.DataFrame({"OOF F1 (official folds)": curve,
              "share of 1s on test": [(proba >= t).mean() for t in curve.index]}).T.round(3)

,0.30,0.32,0.34,0.36,0.38,0.40,0.42,0.44,0.46,0.48,0.50,0.52
OOF F1 (official folds),0.759,0.761,0.765,0.766,0.765,0.763,0.762,0.763,0.762,0.755,0.752,0.749
share of 1s on test,0.880,0.859,0.842,0.825,0.806,0.788,0.756,0.728,0.705,0.689,0.662,0.638


In [9]:
prediction_v07 = pd.DataFrame({"ID": test.index, TARGET: (proba >= V03_THRESHOLD).astype(int)})
check_submission(prediction_v07)

version_v07 = submit(
    prediction_v07,
    notebook="submission_v06_v07_nico",
    model="V06 model (logistic + month + seasonal + L1 C=0.05) with V03's threshold",
    cv_f1=f1_score(y.iloc[last], oof.iloc[last] >= V03_THRESHOLD),
    notes=(f"threshold {V03_THRESHOLD:.2f} fixed to V03's (own CV rule gives {expected['final threshold']:.2f}; "
           f"OOF F1 curve flat 0.34-0.46); isolates the seasonal features: compare with V03 (same threshold) "
           f"and V06 (same model); CV F1 = fold 3 at this threshold; {prediction_v07[TARGET].mean():.0%} predicted 1"),
)
versions = {"V06": version_v06, "V07": version_v07}
versions

V07 saved as submissions\ML_Group_41_Version07.csv and logged:
  Version: 07 | Date: 2026-10-22 | Author: Bosch | Notebook / model: submission_v06_v07_nico · V06 model (logistic + month + seasonal + L1 C=0.05) with V03's threshold | CV F1: 0.796 | Notes: threshold 0.43 fixed to V03's (own CV rule gives 0.35; OOF F1 curve flat 0.34-0.46); isolates the seasonal features: compare with V03 (same threshold) and V06 (same model); CV F1 = fold 3 at this threshold; 74% predicted 1
Next: publish(7) to reserve the version, then upload the file on Kaggle.


{'V06': 6, 'V07': 7}

# <font color='#E8800A'>5. Publish and upload</font> <a class="anchor" id="publish"></a>
[Back to TOC](#toc)

**Reserve both versions on GitHub** (commit + push of the CSVs and the log only), **then** upload the printed files on Kaggle:

In [10]:
# versions = {name: publish(version) for name, version in versions.items()}

After the uploads, record the two public scores and share them:

In [11]:
# record_kaggle_score(versions["V06"], 0.0)  # <- public score of V06
# record_kaggle_score(versions["V07"], 0.0)  # <- public score of V07
# publish(versions["V07"])                   # one publish shares the log with both scores

### How to read the results
- **V07 vs V03** (same threshold, only `seasonal` added): above → the customer's own seasonality transfers to Kaggle.
- **V06 vs V07** (same model, only the threshold changes): tells whether predicting more 1s (84% vs 74%) pays on test.
- The public leaderboard is noisy (±0.02–0.03, `submission_v03_v04_ludovico`): read differences under ~0.01 as ties, and keep `seasonal` on the strength of the validation (ROC AUC 3/3 folds up) unless Kaggle is clearly worse.